In [ ]:
import importlib.util
import subprocess
import sys

REPO_URL = 'https://github.com/manicko/mko_data_cleaner.git'
GIT_REF = 'master'  # selected ref (pinned commit SHA, tag, or branch)

# Optional non-default override examples (uncomment to use):
# GIT_REF = "0743c3d21ed5ac6291ab0a96dfab26db0664c05f"  # pinned delivery commit
# GIT_REF = "vX.Y.Z"  # use only if a tag exists on the remote

FORCE_REINSTALL = True  # set True to reinstall even if the package is already installed

package_name = "mko_data_cleaner"
install_spec = f"git+{REPO_URL}@{GIT_REF}"
print(f"Target: {install_spec}")

installed = importlib.util.find_spec(package_name) is not None
if FORCE_REINSTALL:
    print("FORCE_REINSTALL=True -> installation required.")
    do_install = True
elif installed:
    print(f"Package '{package_name}' is already installed; skipping installation.")
    print("To update or change ref, set FORCE_REINSTALL=True (or reinstall manually), then restart the kernel.")
    do_install = False
else:
    print(f"Package '{package_name}' is not installed; installation required.")
    do_install = True

if do_install:
    if "get_ipython" in globals():
        get_ipython().run_line_magic("pip", f'install --upgrade "{install_spec}"')
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", install_spec])
    print("Install finished. Restart the kernel before continuing if packages/imports changed.")
else:
    print("No install needed.")


## Импорт зависимостей

In [ ]:
from pathlib import Path

import polars as pl
from mko_data_cleaner import process_data, ReportResult
import mko_data_cleaner

print("mko_data_cleaner", mko_data_cleaner.__version__)
print("process_data ->", process_data.__doc__.strip().splitlines()[0])

## Настройка путей и файлов отчета

In [ ]:
# Windows example (comment only):
# import os
# WORKSPACE_ROOT = Path(os.environ["LOCALAPPDATA"]) / "mko_get_mediascope_data" / "mko_get_mediascope_data" / "settings" / "connections" / "mediascope.json"
# WORKSPACE_ROOT = Path(platformdirs.user_config_dir(APP_NAME)) / "settings" /
# Example override to user config/external path (Windows example, comment only):
# WORKSPACE_ROOT = Path(r"%LOCALAPPDATA%\mko_get_mediascope_data\mko_get_mediascope_data\")

# Portable workspace root (override as needed).
WORKSPACE_ROOT = Path.cwd()

SETTINGS_WORKSPACE = WORKSPACE_ROOT
REPORT_PATH = WORKSPACE_ROOT.parent / "theiss"
DICT_PATH = REPORT_PATH / "merged_dictionary.csv"


print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("SETTINGS_WORKSPACE:", SETTINGS_WORKSPACE)
print("REPORT_PATH:", REPORT_PATH)
print("DICT_PATH:", DICT_PATH)

In [ ]:
# Minimal completeness checks
required = [
    SETTINGS_WORKSPACE / "settings" / "app_config.json5",
    SETTINGS_WORKSPACE / "settings" / "log_config.json5",
    REPORT_PATH,
    DICT_PATH,
    
]
missing = [str(p) for p in required if not p.exists()]
print("Missing required paths:", missing if missing else "none")


## Run the pipeline

In [ ]:
for report_dir in REPORT_PATH.iterdir():
    if report_dir.is_dir():
        try:
            print(report_dir.name)  # Prints just the folder name        
            result: ReportResult = process_data(report_dir, settings_dir=SETTINGS_WORKSPACE, dict_path=DICT_PATH)
            print(result)
        except Exception as exc:
            # Do not print secrets/auth payloads; show exception type and non-sensitive message.
            print("RUN STATUS: FAILED - no success claim.")
            print(f"{type(exc).__name__}: {exc}")

## Inspect the result

`ReportResult` exposes `rows_loaded`, `rows_exported`, `timings` (`{import, mapping, export}`), `export_dir`, `null_data_files`, `import_dir`, `report_dir`, `db_path`, plus timing/duration fields.

In [ ]:
print("rows_loaded:", result.rows_loaded)
print("rows_exported:", result.rows_exported)
print("timings:", result.timings)
print("export_dir:", result.export_dir)
print("null_data_files:", result.null_data_files)
clean = next(result.export_dir.glob("*.csv.gz"))
print("clean_data sample:"); display(pl.read_csv(clean, separator=";").head())
if result.null_data_files:
    nd = result.null_data_files[0]
    print("null_data sample:"); display(pl.read_csv(nd, separator=";").head())